# Appendix: ablations

* `basc_temp_ablation_headline.png`: BASC-L (k=5) with probe pools sampled at
  T = 0.5 / 0.7 / 0.9 (Qwen3-1.7B, Qwen3-4B-Thinking-2507; mean over datasets and seeds).
  Inputs: `results/temp_ablation_T{0.5_k20,0.7_k100,0.9_k20}.csv`.
* `basc_semantic_vs_exact.png`: exact-match vs. semantic agreement for BASC-L
  on the QA datasets. Input: `results/main.csv`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from paper_utils import DATASETS_MAP, FIGURES_DIR, RESULTS_DIR, promote_semantic

plt.style.use('seaborn-v0_8')
pal = sns.color_palette("deep")

## Probe temperature

In [ ]:
MODELS = ["Qwen/Qwen3-1.7B", "Qwen/Qwen3-4B-Thinking-2507"]
TEMPS = [0.5, 0.7, 0.9]
METHOD = "basc_k5"          # BASC-L (k=5), as reported in the paper
METRICS = ["ECE1", "ECE2", "Brier", "AUROC"]
LOWER_BETTER = {"ECE1", "ECE2", "Brier"}

raw = pd.concat([pd.read_csv(RESULTS_DIR / f"temp_ablation_T{t}_k{k}.csv")
                 for t, k in [(0.5, 20), (0.7, 100), (0.9, 20)]], ignore_index=True)
temp_df = promote_semantic(raw, by=("model", "dataset", "pool_temp"))

def _stats(model, metric):
    """Mean and SEM across datasets x seeds, per pool temperature."""
    sub = temp_df[(temp_df["method"] == METHOD) & (temp_df["model"] == model)]
    g = sub.groupby("pool_temp")[metric].agg(["mean", "std", "count"]).reindex(TEMPS)
    g["sem"] = g["std"] / np.sqrt(g["count"].clip(lower=1))
    return g

fig, axes = plt.subplots(len(MODELS), len(METRICS), figsize=(3.2 * len(METRICS), 2.7 * len(MODELS)), sharex=True)
for r, model in enumerate(MODELS):
    for c, metric in enumerate(METRICS):
        ax = axes[r, c]
        st = _stats(model, metric)
        means, sems = st["mean"].values, st["sem"].values
        x = np.arange(len(TEMPS))
        ax.bar(x, means, yerr=sems, color=pal[:3], linewidth=1.2, capsize=3,
               error_kw={"ecolor": "#333", "lw": 0.9})
        pad = max(float(np.nanmax(means) - np.nanmin(means)), 1e-3) * 0.15
        for xi, v in zip(x, means):
            ax.text(xi + 0.25, v + pad, f"{v:.3f}", ha="center", va="bottom", fontsize=9, color="#0b0b0b")
        ax.set_xticks(x)
        ax.set_xticklabels([f"T={t}" for t in TEMPS])
        if r == 0:
            ax.set_title(f"{metric} {'↓' if metric in LOWER_BETTER else '↑'}", fontsize=13)
        if c == 0:
            ax.set_ylabel(model.split("/")[-1], fontsize=12)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        ax.grid(axis="y", alpha=0.25)
fig.suptitle("BASC-L (k=5) - Sampling Temperature Ablation", fontsize=13, y=0.97)
fig.tight_layout()
plt.savefig(FIGURES_DIR / "basc_temp_ablation_headline.png", dpi=300, bbox_inches="tight")
plt.show()

## Exact-match vs. semantic agreement (QA)

In [ ]:
QA_DATASETS = ["sciq", "trivia_qa", "webq"]
KS = ["2", "5"]
INK, INK_MUTED, AXIS_MUTED, GRID = "#0b0b0b", "#52514e", "#898781", "#e1e0d9"

df = pd.read_csv(RESULTS_DIR / "main.csv")
qa = df[df["dataset"].isin(QA_DATASETS) & df["model"].isin(df.loc[df["method"].str.startswith("basc_sem_"), "model"])]

fig, axes = plt.subplots(1, len(QA_DATASETS), figsize=(13, 3), sharey=False)
for ax, ds in zip(axes, QA_DATASETS):
    sub = qa[qa["dataset"] == ds]
    exact_vals, sem_vals, exact_err, sem_err = [], [], [], []
    for k in KS:
        em = sub[sub["method"] == f"basc_k{k}"]["ECE2"]
        sm = sub[sub["method"] == f"basc_sem_k{k}"]["ECE2"]
        exact_vals.append(em.mean()); sem_vals.append(sm.mean())
        exact_err.append(em.std() / np.sqrt(max(len(em), 1)))
        sem_err.append(sm.std() / np.sqrt(max(len(sm), 1)))
    top = max(max(exact_vals), max(sem_vals))
    x, w = np.arange(len(KS)), 0.36
    ax.bar(x - w / 2, exact_vals, w, yerr=exact_err, color=pal[0], linewidth=2, label="exact",
           error_kw={"ecolor": INK_MUTED, "capsize": 3, "lw": 1})
    ax.bar(x + w / 2, sem_vals, w, yerr=sem_err, color=pal[1], linewidth=2, label="semantic",
           error_kw={"ecolor": INK_MUTED, "capsize": 3, "lw": 1})
    for xi, v in zip(x, exact_vals):
        ax.text(xi - w / 2, v + top * 0.02, f"{v:.3f}", ha="center", va="bottom", fontsize=8, color=INK)
    for xi, v in zip(x, sem_vals):
        ax.text(xi + w / 2, v + top * 0.02, f"{v:.3f}", ha="center", va="bottom", fontsize=8, color=INK)
    ax.set_xticks(x); ax.set_xticklabels([f"k={k}" for k in KS], color=INK, fontsize=16)
    for side in ("top", "right"): ax.spines[side].set_visible(False)
    for side in ("left", "bottom"): ax.spines[side].set_color(AXIS_MUTED)
    ax.tick_params(colors=INK_MUTED, labelsize=12)
    ax.yaxis.grid(True, color=GRID, lw=0.8); ax.set_axisbelow(True)
    ax.set_ylim(0, top * 1.5)
    if ax is axes[0]:
        ax.set_ylabel("ECE₂", color=INK, fontsize=14)
    ax.set_title(DATASETS_MAP[ds], color=INK, fontsize=14)
axes[0].legend(loc="upper right", fontsize=13, frameon=False, labelcolor=INK, ncol=2)
fig.suptitle("Exact-match vs Semantic-match Agreement", fontsize=14, color=INK, y=0.95)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "basc_semantic_vs_exact.png", dpi=600, bbox_inches="tight")
plt.show()